# KALIA v0.2.0 — export the checkpoint and prove the export is faithful

A release is only as good as the check that it is the thing it claims to be. The
training checkpoint is **not** a loadable model repo: it carries optimizer state, our
internal tensor names, and no config. So this kernel does the conversion and then
**re-scores the written `model.safetensors`** under the registered deterministic
protocol.

| step | value |
|---|---|
| 1 | copy the checkpoint and the code, out of the code dataset |
| 2 | write `model.safetensors`, `config.json`, `generation_config.json`, `model.py`, `sample.py`, `LICENSE` |
| 3 | assert the tensors are bitwise-identical to the checkpoint |
| 4 | assert the tensor set matches the already-shipped v0.1.2 layout |
| 5 | load the written tensors into the repo's own `model.py` and re-score |
| 6 | compare against the registered **2.8248** and stop if it does not match |

CPU, no GPU quota, and nothing is downloaded to the developer machine.

The config is **derived from the checkpoint's own** `config['model']`, never typed by
hand, so the published architecture cannot drift from the trained one.

In [ ]:
!pip install -q "safetensors" "tiktoken<1" pyyaml numpy pandas
import glob, json, os, shutil, sys
root = sorted(glob.glob('/kaggle/input/datasets/subhajitlucky/kalia-code-dev'))[0]
work = '/kaggle/working/kalia'
if not os.path.exists(work):
    shutil.copytree(root, work)
for f in ['tools', 'eval']:
    d = f'{root}/{f}'
    if os.path.isdir(d):
        shutil.copytree(d, f'{work}/{f}', dirs_exist_ok=True)
ckpt = sorted(glob.glob(
    '/kaggle/input/notebooks/subhajitlucky/kalia-train-v020/**/ckpt.pt', recursive=True))
assert ckpt, 'attach the kalia-train-v020 kernel output'
val_bin = sorted(glob.glob(
    '/kaggle/input/notebooks/subhajitlucky/kalia-prep-v2b/**/val.bin', recursive=True))
assert val_bin, 'attach the kalia-prep-v2b kernel output for val.bin'
os.chdir(work)
print('ckpt:', ckpt[0])
print('val.bin:', val_bin[0])

In [ ]:
import torch
from export_hf_model import build_config, write_safetensors

OUT = '/kaggle/working/release'
os.makedirs(OUT, exist_ok=True)
ck = torch.load(ckpt[0], map_location='cpu', weights_only=False)
step, tokens = int(ck['step']), int(ck['tokens'])
print('step', step, '| tokens', f'{tokens:,}')
assert step == 4770, f'expected step 4770, got {step}'

cfg = build_config(ck['config']['model'])
open(f'{OUT}/config.json', 'w').write(json.dumps(cfg, indent=2) + '\n')
open(f'{OUT}/generation_config.json', 'w').write(json.dumps({
    'max_new_tokens': 80, 'temperature': 0.8, 'top_k': 200,
    'bos_token_id': 50256, 'eos_token_id': 50256}, indent=2) + '\n')
n = write_safetensors(ck['model'], step, tokens, f'{OUT}/model.safetensors')
for f in ['model.py', 'sample.py', 'LICENSE']:
    shutil.copy(f'{work}/{f}', f'{OUT}/{f}')
# The card is shipped in the code dataset and published as README.md, so the
# kernel that verifies the weights also publishes the card describing them.
shutil.copy(f'{work}/hf-model-card-v020.md', f'{OUT}/README.md')
print(f'wrote {n} tensors ->', sorted(os.listdir(OUT)))
assert os.path.exists(f'{OUT}/README.md'), 'card was not staged'

In [ ]:
# Step 3: the conversion must not have altered a single bit.
from safetensors.torch import load_file, safe_open
exp = load_file(f'{OUT}/model.safetensors')
bad = [k for k in exp if not torch.equal(exp[k], ck['model'][k])]
assert not bad, f'export altered {len(bad)} tensors, e.g. {bad[:3]}'
print(f'{len(exp)} tensors bitwise-identical to the checkpoint')
with safe_open(f'{OUT}/model.safetensors', framework='pt') as f:
    print('metadata:', f.metadata())

In [ ]:
# Step 4: the released repo must have the same shape as the one already shipped.
import urllib.request
try:
    url = ('https://huggingface.co/kalia-lm/kalia-v012/raw/main/model.safetensors')
    with safe_open(url, framework='pt') as f:
        shipped = set(f.keys())
    print('shipped v0.1.2 tensors:', len(shipped))
    print('same set as ours:', shipped == set(exp))
    print('only in shipped:', sorted(shipped - set(exp))[:5])
    print('only in ours   :', sorted(set(exp) - shipped)[:5])
except Exception as exc:
    print('could not fetch v0.1.2 for comparison:', type(exc).__name__, exc)

In [ ]:
# Step 5: score the WRITTEN tensors under the registered protocol.
import math
import numpy as np
import tiktoken
from model import GPT, GPTConfig
from data import TokenDataset
from export_hf_model import PROTOCOL

published = json.load(open(f'{OUT}/config.json'))
mcfg = GPTConfig(**{k: published[k] for k in
                   ('vocab_size', 'n_layer', 'n_head', 'n_embd',
                    'context_len', 'dropout', 'qk_norm', 'logit_softcap')})
model = GPT(mcfg)
state = load_file(f'{OUT}/model.safetensors')
missing, unexpected = model.load_state_dict(state, strict=False)
assert not unexpected, f'unexpected tensors: {unexpected[:5]}'
assert not missing, f'missing tensors: {missing[:5]}'
assert torch.equal(state['lm_head.weight'], state['tok_emb.weight']), 'head is not tied'
print('published tensors load into the repo model cleanly')
model.eval()

ds = TokenDataset(val_bin[0], mcfg.context_len)
g = torch.Generator().manual_seed(PROTOCOL['seed'])
total, ntok = 0.0, 0
with torch.no_grad():
    for _ in range(PROTOCOL['batches']):
        x, y = ds.get_batch(PROTOCOL['batch_size'], torch.device('cpu'), g)
        _, loss = model(x, y)
        total += loss.item() * y.numel()
        ntok += int(y.numel())
val = total / ntok
bpb = val / math.log(2) / 4.4086
print(f'\nexported weights score {val:.4f} / {bpb:.4f} bpB over {ntok:,} tokens')

In [ ]:
REGISTERED_VAL, REGISTERED_BPB, TOL = 2.8248, 0.9244, 5e-4
print('=== release gate ===')
print(f'  registered : {REGISTERED_VAL} / {REGISTERED_BPB}')
print(f'  exported   : {val:.4f} / {bpb:.4f}')
ok = abs(val - REGISTERED_VAL) <= TOL
print(f'  delta      : {val - REGISTERED_VAL:+.4f}  (tolerance {TOL})')
res = {'val_loss': round(val, 4), 'bpb': round(bpb, 4), 'tokens': ntok,
       'registered': REGISTERED_VAL, 'matches': bool(ok),
       'step': step, 'train_tokens': tokens}
json.dump(res, open('/kaggle/working/export_verification.json', 'w'), indent=2)
if not ok:
    print('\nFAIL: the exported weights do not reproduce the registered number.')
    print('Do not publish. A conversion that changes the score is a different model.')
else:
    print('\nPASS: the artifact to be published scores exactly what was pre-registered.')

In [ ]:
# Publish the artifact this kernel just verified.
#
# Deliberately the *same* kernel: the file that scored 2.8248 is the file that
# gets pushed, so there is no window in which a different copy could be uploaded
# instead. That has been the failure mode four times in this project already --
# four kernels died behind green uploads, and once a model-card push silently
# created a stray directory instead of updating the file.
#
# Requires HF_TOKEN as a Kaggle secret on this notebook. It does nothing at all
# unless the gate above passed.
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
res = json.load(open('/kaggle/working/export_verification.json'))
assert res['matches'], 'release gate did not pass; refusing to publish'
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
REPO = 'kalia-lm/kalia-v020'
FILES = ['model.safetensors', 'config.json', 'generation_config.json',
         'model.py', 'sample.py', 'LICENSE', 'README.md']
for name in FILES:
    src = f'{OUT}/{name}'
    if not os.path.exists(src):
        print('  missing', name, '-- skipped')
        continue
    api.upload_file(path_or_fileobj=src, path_in_repo=name, repo_id=REPO,
                    repo_type='model',
                    commit_message=f'release: v0.2.0, step 4770, verified at 2.8248')
    print('  uploaded', name, os.path.getsize(src), 'bytes')
print('uploaded to', REPO, '-- repo left PRIVATE; flip public only after review')

In [ ]:
# Read the published repo back and confirm the weights are the verified ones.
from safetensors.torch import safe_open
import urllib.request
try:
    with safe_open(f'{OUT}/model.safetensors', framework='pt') as f:
        meta = f.metadata()
    print('local metadata:', meta)
    assert meta['step'] == '4770'
    url = f'https://huggingface.co/{REPO}/raw/main/model.safetensors'
    req = urllib.request.Request(url, headers={'Range': 'bytes=0-0'})
    with urllib.request.urlopen(req, timeout=60) as resp:
        print('remote responds:', resp.status, '|', resp.headers.get('Content-Length'), 'bytes')
except Exception as exc:
    print('could not confirm the remote copy:', type(exc).__name__, exc)

If this cell prints **FAIL**, nothing from this kernel may be uploaded. The gate
compares the artifact against the number committed to in
`docs/preregistrations/2026-09-28-v020-final-evaluation.md` before the results existed,
which is the only reason the check means anything.

On PASS, the remaining release steps are: write the v0.2.0 model card (which must
state that it fails its own stability rule on two of five tasks), upload the seven
files, and only then flip the repo public.